In [1]:

import os

from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='Qwen/Qwen3-Omni-30B-A3B-Instruct',
    base_url=os.environ.get('OPENAI_BASE_URL'),
    api_key=os.environ.get('OPENAI_API_KEY')
)

D:\Work\Langchain-learn-demo\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# 基于内存的聊天历史

In [76]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    # 系统提示
    ('system', '你是一个乐于助人的助手。尽你所能回答所有的问题。提供的聊天历史包含与你对话用户的相关信息'),
    # 聊天历史
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    # 用户输入
    ('human', '{input}')
])

In [3]:
from langchain_core.chat_history import InMemoryChatMessageHistory

chain = prompt | model

store = {}


def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

In [78]:
from langchain_core.runnables import RunnableWithMessageHistory

# 创建带有聊天历史的chain
chain = RunnableWithMessageHistory(
    prompt | model,
    get_session_history,
    input_messages_key='input',
    history_messages_key='chat_history'
)

In [79]:
config = {
    "configurable": {
        "session_id": "user123"
    }
}
chain.invoke({'input': '你好'}, config=config)

AIMessage(content='你好！有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 8, 'prompt_tokens': 39, 'total_tokens': 47, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9eb59e5c42aa428a52fcc067fa73', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9eb5-a3e8-73e0-b731-8cfae3593cdc-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 8, 'total_tokens': 47, 'input_token_details': {}, 'output_token_details': {}})

In [6]:
chain.invoke({'input': '你是谁'}, config=config)

AIMessage(content='我是阿里巴巴集团旗下的通义实验室自主研发的多模态超大规模语言模型，我叫通义千问Omni。有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 32, 'prompt_tokens': 63, 'total_tokens': 95, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9e5e591fd2000ea80fa99ed46649', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9e5e-5fc7-7243-8b73-026b6354c656-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 63, 'output_tokens': 32, 'total_tokens': 95, 'input_token_details': {}, 'output_token_details': {}})

In [7]:
chain.invoke({'input': '我刚才说的第一句话是什么'}, config=config)

AIMessage(content='你刚才说的第一句话是：“你好”。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 111, 'total_tokens': 120, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9e5e5adc32cca100e98281729579', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9e5e-6179-7f01-9cf9-e1926d6aeba9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 111, 'output_tokens': 9, 'total_tokens': 120, 'input_token_details': {}, 'output_token_details': {}})

In [8]:
for message in store['123'].messages:
    print(type(message), message.content)

<class 'langchain_core.messages.human.HumanMessage'> 你好
<class 'langchain_core.messages.ai.AIMessage'> 你好！很高兴见到你。有什么我可以帮助你的吗？
<class 'langchain_core.messages.human.HumanMessage'> 你是谁
<class 'langchain_core.messages.ai.AIMessage'> 我是阿里巴巴集团旗下的通义实验室自主研发的多模态超大规模语言模型，我叫通义千问Omni。有什么我可以帮助你的吗？
<class 'langchain_core.messages.human.HumanMessage'> 我刚才说的第一句话是什么
<class 'langchain_core.messages.ai.AIMessage'> 你刚才说的第一句话是：“你好”。


# 使用外部存储保存聊天历史

In [67]:
from sqlalchemy.ext.asyncio import create_async_engine
from langchain_community.chat_message_histories import SQLChatMessageHistory

# 使用异步调用
async_engine = create_async_engine("sqlite+aiosqlite:///data/chat_history.db")


def get_session_history_sql(session_id):
    return SQLChatMessageHistory(
        session_id=session_id,
        connection=async_engine
    )


# 创建带有聊天历史的chain
chain = RunnableWithMessageHistory(
    prompt | model,
    get_session_history_sql,
    input_messages_key='input',
    history_messages_key='chat_history'
)
await chain.ainvoke({'input': '你好'}, config=config)
await chain.ainvoke({'input': '你是谁'}, config=config)
await chain.ainvoke({'input': '我刚才说的第一句话是什么'}, config=config)

AIMessage(content='你说的第一句话是：“你好”。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 149, 'total_tokens': 156, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9eb2375a08d366ae01f3a503b533', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9eb2-3d94-7d73-a342-d974ae916bfa-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 149, 'output_tokens': 7, 'total_tokens': 156, 'input_token_details': {}, 'output_token_details': {}})

# 剪辑和摘要上下文

In [40]:

from langchain_core.messages import BaseMessage

summary_len = 5


async def summarize_context(current_input):
    session_id = current_input.get('config', {}).get("configurable", {}).get('session_id')
    if not session_id:
        raise ValueError('session_id is required')
    chat_history = get_session_history_sql(session_id)
    stored_messages = await chat_history.aget_messages()
    if len(stored_messages) <= summary_len:
        return False
    print("开始摘要历史对话")
    last_two_messages = stored_messages[-summary_len:]
    messages_to_be_summarized = stored_messages[:-summary_len]

    summarize_prompt = ChatPromptTemplate.from_messages([
        ('system', '请将下面消息进行剪辑和摘要，并返回结果'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '请生成包含上述对话核心内容的摘要，暴露重要的事实和决策'),
    ])

    summarize_chain = summarize_prompt | model
    summary_messages: BaseMessage = await summarize_chain.ainvoke({
        'chat_history': messages_to_be_summarized,
    })
    await chat_history.aclear()
    await chat_history.aadd_message(summary_messages)
    await chat_history.aadd_messages(last_two_messages)
    return True

In [64]:

from langchain_core.runnables import RunnablePassthrough

await get_session_history_sql(config['configurable']['session_id']).aclear()

# 创建带有聊天历史的chain
chain = (
        RunnablePassthrough.assign(messages_summarized=summarize_context) |
        RunnableWithMessageHistory(
            prompt | model,
            get_session_history_sql,
            input_messages_key='input',
            history_messages_key='chat_history'
        )
)
await chain.ainvoke({'input': '你好,今天天气很晴朗', 'config': config}, config=config)
await chain.ainvoke({'input': '你是谁', 'config': config}, config=config)
await chain.ainvoke({'input': '今天天气怎么样', 'config': config}, config=config)

KeyError: "Input to ChatPromptTemplate is missing variables {'summary'}.  Expected: ['input', 'summary'] Received: ['input', 'config', 'messages_summarized', 'chat_history']\nNote: if you intended {summary} to be part of the string and not a variable, please escape it with double curly braces like: '{{summary}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PROMPT_INPUT "

In [49]:
await get_session_history_sql(config['configurable']['session_id']).aget_messages()

[AIMessage(content='摘要：对话双方确认当天天气晴朗，并对此表达了积极评价。发言人建议充分利用好天气，可能倾向于安排户外活动。未提及具体决策，但隐含了对当日活动计划的讨论或调整打算。关键事实为天气状况良好，情绪倾向积极。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 84, 'total_tokens': 141, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9e8ae23332afc5716369c568d7ec', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9e8a-e8d8-7b03-ae5e-331a0a998857-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 57, 'total_tokens': 141, 'input_token_details': {}, 'output_token_details': {}}),
 HumanMessage(content='你是谁', additional_kwargs={}, response_metadata={}),
 AIMessage(content='我是来自阿里巴巴集团通义实验室的通义千问Omni，是阿里巴巴集团自主研发的多模态超大规模语言模型，旨在为用户提供高效、优质的信息服务与交流体验。有什么我可以帮助你的吗？', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'co

- 上面的摘要思路是从历史聊天记录下手，导致历史记录丢失，下面进行优化

In [83]:
from langchain_core.messages import HumanMessage

summary_len = 5


async def summarize_context_v2(current_input):
    session_id = current_input.get('config', {}).get("configurable", {}).get('session_id')
    if not session_id:
        raise ValueError('session_id is required')
    chat_history = get_session_history_sql(session_id)
    stored_messages = await chat_history.aget_messages()
    if len(stored_messages) <= summary_len:
        return {"original_messages": stored_messages, "summary": None}
    print("开始摘要历史对话")
    last_two_messages = stored_messages[-summary_len:]
    messages_to_be_summarized = stored_messages[:-summary_len]

    summarize_prompt = ChatPromptTemplate.from_messages([
        ('system', '请将下面消息进行剪辑和摘要，并返回结果'),
        MessagesPlaceholder(variable_name='chat_history'),
        ('human', '请生成包含上述对话核心内容的摘要，暴露重要的事实和决策'),
    ])

    summarize_chain = summarize_prompt | model
    summary_message = await summarize_chain.ainvoke({
        'chat_history': messages_to_be_summarized,
    })
    return {
        "original_messages": last_two_messages,  # 保留的原始消息
        "summary": summary_message
    }


prompt = ChatPromptTemplate.from_messages([
    ('system', "你是一个乐于助人的助手。尽你所能回答所有问题。摘要：{summary}"),  # 动态注入系统消息
    MessagesPlaceholder(variable_name='chat_history', optional=True),
    ('human', '{input}')
])


async def invoke_and_save(context):
    session_id = context.get('config', {}).get("configurable", {}).get('session_id')
    if not session_id:
        raise ValueError('session_id is required')
    messages_summarized = context.get('messages_summarized', {})
    summary = messages_summarized.get('summary', '无摘要')
    chat_history = messages_summarized.get('original_messages')
    user_input = context.get('input')
    invoke_chain = prompt | model
    print(prompt.invoke({
        'input': user_input,
        'chat_history': chat_history,
        'summary': summary
    }))
    result = await invoke_chain.ainvoke({
        'input': user_input,
        'chat_history': chat_history,
        'summary': summary
    })
    chat_history_sql = get_session_history_sql(session_id)
    await chat_history_sql.aadd_message(HumanMessage(content=user_input))
    await chat_history_sql.aadd_message(result)
    return result



In [87]:
chain = (
        RunnablePassthrough.assign(messages_summarized=summarize_context_v2) |
        RunnablePassthrough.assign(
            input=lambda x: x['input'],
            system_message=lambda x: x['messages_summarized']['summary'] or '无摘要',
            chat_history=lambda x: x['messages_summarized']['original_messages']
        ) | invoke_and_save

)
await get_session_history_sql(config['configurable']['session_id']).aclear()
await chain.ainvoke({'input': '你好，我是蔡锷?', "config": {"configurable": {"session_id": "user123"}}},
                    config={"configurable": {"session_id": "user123"}})
await chain.ainvoke({'input': '我的名字叫什么?', "config": {"configurable": {"session_id": "user123"}}},
                    config={"configurable": {"session_id": "user123"}})
await chain.ainvoke({'input': '我喜欢打球!', "config": {"configurable": {"session_id": "user123"}}},
                    config={"configurable": {"session_id": "user123"}})
await chain.ainvoke(
    {'input': '用我的名字，写一篇短文，不超过200个字？', "config": {"configurable": {"session_id": "user123"}}},
    config={"configurable": {"session_id": "user123"}})

messages=[SystemMessage(content='你是一个乐于助人的助手。尽你所能回答所有问题。摘要：None', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我是蔡锷?', additional_kwargs={}, response_metadata={})]
messages=[SystemMessage(content='你是一个乐于助人的助手。尽你所能回答所有问题。摘要：None', additional_kwargs={}, response_metadata={}), HumanMessage(content='你好，我是蔡锷?', additional_kwargs={}, response_metadata={}), AIMessage(content='您好，我无法确认您的身份。作为一个AI助手，我无法验证用户的真实身份。如果您是蔡锷大将军（历史人物），那么我是无法与您直接对话的；如果您是现代的蔡锷先生/女士，请您明示您的身份信息以便我更好地为您提供帮助。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 60, 'prompt_tokens': 37, 'total_tokens': 97, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9eb897d1e90d4cc7aad8cfacc46c', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9eb8-9d72-7000-960c-c8d7c771d541-0', tool_calls=[], invalid_tool_calls=[], usage_metad

AIMessage(content='蔡锷自幼聪颖，志存高远，尤喜运动。闲暇之余，他常至球场挥洒汗水，球风刚健，步伐敏捷。他不但 kỹ thuật 精湛，更以坚韧不拔的精神带动队友。球场如人生战场，蔡锷以球会友，用拼搏诠释努力，凭毅力赢得尊重。每逢比赛，他总以热血激昂的姿态诠释“锲而不舍”的信念。在篮球的每一次跃起和扣篮中，蔡锷点燃了青春最壮美的火焰。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 115, 'prompt_tokens': 458, 'total_tokens': 573, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'Qwen/Qwen3-Omni-30B-A3B-Instruct', 'system_fingerprint': '', 'id': '019d9eb89fd7c44623b378c171028650', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d9eb8-a612-75d2-9845-59bd501af292-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 458, 'output_tokens': 115, 'total_tokens': 573, 'input_token_details': {}, 'output_token_details': {}})